# Mod-12 Einzelmodul-Generator (SageMath Native Version)

Dieses Notebook erzeugt alle Zahlen ≤ n, deren Primfaktoren ausschließlich aus genau einer der mod-12 Klassen {1, 5, 7, 11} stammen.

**Funktionen:**
- Nutzt parallele Verarbeitung mit SageMath's `@parallel` Decorator
- Nutzt SageMath's optimierte Primzahl-Funktionen (`prime_range()`)
- Nutzt SageMath's optimierte Faktorisierung (`factor()`)
- Generiert CSV-Dateien sortiert nach Wert

**Kompatibilität:** SageMath 10.8.1

---

## 📋 Anleitung zur Ausführung

**Wichtig:** Dieses Notebook benötigt den **SageMath-Kernel**!

### Schritte:
1. **Stellen Sie sicher, dass der SageMath-Kernel aktiv ist** (oben rechts im Notebook)
2. **Führen Sie die Zellen der Reihe nach aus** (Shift+Enter für jede Zelle)
3. **Beginnen Sie mit Zelle 1** (SageMath-Imports)
4. **Führen Sie die restlichen Zellen nacheinander aus**

### Ausgabe:
Das Programm erzeugt eine CSV-Datei:
- `mod12_single_module_by_value.csv` - sortiert nach numerischem Wert

In [ ]:
# SageMath Imports
from sage.all import *
import csv
import sys
import time

print("✓ SageMath erfolgreich geladen!")
sage_version = version()
print(f"SageMath Version: {sage_version}")

# Versionsprüfung für SageMath 10.8.1
if "10.8.1" in sage_version or "10.8" in sage_version:
    print("✓ Kompatible SageMath-Version erkannt")
else:
    print("⚠️  Warnung: Dieses Notebook wurde für SageMath 10.8.1 entwickelt")
    print("   Es könnte mit anderen Versionen funktionieren, ist aber nicht getestet")

print(f"Python Version: {sys.version}")
sys.stdout.flush()

✓ SageMath erfolgreich geladen!
SageMath Version: SageMath version 10.8, Release Date: 2025-12-18
Python Version: 3.13.7 (main, Dec 26 2025, 08:28:22) [Clang 17.0.0 (clang-1700.3.19.1)]


## Konfiguration

In [2]:
# Konfiguration
GROUP_ORDER = [1, 5, 7, 11]
DEFAULT_N = 1000
OUT_PREFIX = "mod12_single_module"

# Obergrenze n (kann hier geändert werden)
n = DEFAULT_N
out_prefix = OUT_PREFIX

print(f"Konfiguration:")
print(f"  Obergrenze n = {n}")
print(f"  Ausgabe-Präfix = {out_prefix}")
print(f"  Mod-12 Klassen = {GROUP_ORDER}")

Konfiguration:
  Obergrenze n = 1000
  Ausgabe-Präfix = mod12_single_module
  Mod-12 Klassen = [1, 5, 7, 11]


## Hilfsfunktionen

In [3]:
def get_factorization_string(n):
    """Nutzt SageMath's optimierte Faktorisierung für die String-Darstellung."""
    fac = factor(n)
    return str(fac).replace(" ", "")

print("✓ Hilfsfunktionen definiert")

✓ Hilfsfunktionen definiert


## Parallele Generierungsfunktion

In [4]:
@parallel(ncpus=4)
def generate_for_residue(n, primes, residue):
    """
    Parallele Funktion zur Generierung der Zahlen für eine Restklasse.
    """
    records = []
    # Pre-cache für schnellere Index-Abfrage
    prime_to_idx = {p: i + 1 for i, p in enumerate(primes)}
    
    def dfs(start_i, current_val, current_indices):
        if current_val > 1:
            # Wir berechnen die Faktorisierung erst am Ende oder hier
            # Da wir die Faktoren schon kennen, nutzen wir sie direkt
            fac = factor(current_val)
            
            records.append({
                "group_mod12": residue,
                "value": current_val,
                "factorization": str(fac).replace(" ", ""),
                "indices_str": " ".join(map(str, sorted(current_indices))),
                "len": len(current_indices),
                "min_index": min(current_indices),
                "max_index": max(current_indices),
            })
            
        for i in range(start_i, len(primes)):
            p = primes[i]
            if current_val * p > n:
                break
            
            current_indices.append(i + 1)
            dfs(i, current_val * p, current_indices)
            current_indices.pop()

    dfs(0, 1, [])
    return records

print("✓ Parallele Generierungsfunktion definiert")

✓ Parallele Generierungsfunktion definiert


## Hauptfunktion

In [5]:
def run_generator(n=DEFAULT_N, out_prefix=OUT_PREFIX):
    print(f"=== SageMath Mod-12 Generator ===")
    print(f"Obergrenze: n = {n}")
    print(f"SageMath Version: {version()}")
    
    start_time = time.time()
    
    # 1. Primzahlen generieren
    print(f"Erzeuge Primzahlen bis {n}...")
    primes_up_to_n = prime_range(n + 1)
    
    groups = {r: [] for r in GROUP_ORDER}
    for p in primes_up_to_n:
        r = p % 12
        if r in groups:
            groups[r].append(p)
            
    for r in GROUP_ORDER:
        print(f"  Klasse {r:2d}: {len(groups[r])} Primzahlen")

    # 2. Parallel Zahlen generieren
    print("\nStarte parallele Generierung...")
    # Wir bereiten die Argumente für @parallel vor
    # Format: [(args, kwargs), ...]
    tasks = [((n, groups[r], r), {}) for r in GROUP_ORDER if groups[r]]
    
    all_records = []
    for result in generate_for_residue(tasks):
        # result ist ((n, ps, r), records)
        residue_records = result[1]
        all_records.extend(residue_records)
        print(f"  Klasse {result[0][0][2]:2d}: {len(residue_records)} Zahlen gefunden")

    print(f"\nGesamtanzahl: {len(all_records)} Zahlen erzeugt.")
    
    # 3. Sortieren
    print("Sortiere Ergebnisse...")
    # Sortierung nach Wert
    by_value = sorted(all_records, key=lambda x: (x["value"], x["group_mod12"]))
    
    # 4. CSV Export
    fields = ["group_mod12", "value", "factorization", "indices_str", "len", "min_index", "max_index"]
    
    out_file = f"{out_prefix}_by_value.csv"
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(by_value)
        
    end_time = time.time()
    print(f"✓ Fertig! CSV geschrieben nach: {out_file}")
    print(f"Dauer: {end_time - start_time:.2f} Sekunden")
    
    return all_records, out_file

print("✓ Hauptfunktion definiert")

✓ Hauptfunktion definiert


## Ausführung

Führen Sie diese Zelle aus, um die Generierung zu starten. Sie können die Werte `n` und `out_prefix` in der Konfigurationszelle oben ändern.

In [6]:
# Führe die Generierung aus
# Verwende die Werte aus der Konfigurationszelle
records, output_file = run_generator(n, out_prefix)

print(f"\n✓ Generierung abgeschlossen!")
print(f"  Anzahl gefundener Zahlen: {len(records)}")
print(f"  Ausgabedatei: {output_file}")

=== SageMath Mod-12 Generator ===
Obergrenze: n = 1000
SageMath Version: SageMath version 10.8, Release Date: 2025-12-18
Erzeuge Primzahlen bis 1000...
  Klasse  1: 36 Primzahlen
  Klasse  5: 44 Primzahlen
  Klasse  7: 44 Primzahlen
  Klasse 11: 42 Primzahlen

Starte parallele Generierung...
  Klasse  1: 40 Zahlen gefunden
  Klasse  7: 59 Zahlen gefunden
  Klasse  5: 65 Zahlen gefunden
  Klasse 11: 49 Zahlen gefunden

Gesamtanzahl: 213 Zahlen erzeugt.
Sortiere Ergebnisse...
✓ Fertig! CSV geschrieben nach: mod12_single_module_by_value.csv
Dauer: 0.03 Sekunden

✓ Generierung abgeschlossen!
  Anzahl gefundener Zahlen: 213
  Ausgabedatei: mod12_single_module_by_value.csv
